<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; EDM&plus; Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">EDM&plus; NB05 &mdash; Quotes &amp; Structured Result Store</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Generates 30 business days of realistic price data for 281 instruments so portfolios can be valued and returns calculated.</div>
</div>

<sub>EDM&plus; pack sequence: NB01 &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; <b>NB05</b> &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; NB07</sub>

# NB05: Quotes & Structured Result Store

**What this does:** Generates 30 business days of price data for 281 instruments (~8,430 quotes).

**Business context:** Quotes are the prices used for valuation. Without quotes, you cannot value portfolios or calculate returns. This notebook generates realistic synthetic prices for every equity, bond, and ETF loaded in NB02.

**Verify after running:** Instruments → search Apple → Quotes tab → select scope **edm-training** from the dropdown.

In [ ]:
# Run this cell first — installs required packages (takes ~30 seconds, only needed once per session)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'lusid-sdk', 'lusid-workflow-sdk', 'finbourne-sdk-utilities', 'lusid-drive-sdk', 'lumipy'])
print("✅ Packages installed")

In [ ]:
# ============================================================================
# CREDENTIALS: edit secrets.json (NOT this notebook)
# ============================================================================
# Copy secrets.json.example to secrets.json and fill in your domain + PAT:
#   {
#     "api_url": "https://<YOUR_DOMAIN>.lusid.com/api",
#     "personal_access_token": "<YOUR_PAT>"
#   }
# To get a PAT: LUSID web app → profile icon (top right) → Personal Access Tokens → Create
# (Override the file location with the EDM_SECRETS_PATH environment variable.)

import json as _json, os as _os
_secrets_path = _os.environ.get("EDM_SECRETS_PATH", "secrets.json")
with open(_secrets_path) as _f:
    _secrets = _json.load(_f)
api_url = _secrets["api_url"]
personal_access_token = _secrets["personal_access_token"]

# ============================================================================
# Imports — do not edit below this line
# ============================================================================
import os, json
from datetime import datetime, timedelta, date, timezone
import datetime as dt
import pandas as pd
import pytz

import lusid as lu
import lusid.models as lm

try:
    import lusid_workflow as lwf
    import lusid_workflow.models as wf_models
except ImportError:
    lwf = None
    wf_models = None
    print("⚠️ lusid_workflow not available")

import lumipy as lp

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.4f}".format

# ============================================================================
# Authentication
# ============================================================================
if "<YOUR_DOMAIN>" in api_url or "<YOUR_PAT>" in personal_access_token or not personal_access_token:
    raise ValueError(
        "\n\n⛔ You need to edit the two lines at the top of this cell:\n"
        "   1. Replace <YOUR_DOMAIN> with your LUSID domain name\n"
        "   2. Replace <YOUR_PAT> with your Personal Access Token\n\n"
        "   To get a PAT: LUSID web app → profile icon → Personal Access Tokens → Create")

def get_factory(app='lusid'):
    module = __import__(app)
    # Each service has a different URL path
    url_map = {'lusid': '/api', 'lusid_workflow': '/workflow', 'lusid_drive': '/drive'}
    service_url = api_url.replace('/api', url_map.get(app, '/api'))
    config_loaders = [module.extensions.ArgsConfigurationLoader(
        api_url=service_url, access_token=personal_access_token)]
    return module.extensions.SyncApiClientFactory(config_loaders=config_loaders)

def get_lumi():
    luminesce_url = api_url.replace('/api', '/honeycomb')
    return lp.get_client(access_token=personal_access_token, api_url=luminesce_url)

# Initialise
lusid_factory = get_factory('lusid')

# Verify connection
try:
    api_instance = lusid_factory.build(lu.ApplicationMetadataApi)
    api_response = api_instance.get_lusid_versions()
    domain = api_response.links[0].href
    env_url = domain[0:domain.find('/app/')]
    print(f"✅ Connected to {env_url}")
    print(f"   API Version: {api_response.build_version}")
except Exception as e:
    print(f"⚠️ Connected but could not verify: {e}")

lumi = get_lumi()
print("✅ Luminesce ready")

---
## Generate and Load Quotes

In [ ]:
import random

quotes_api = lusid_factory.build(lu.QuotesApi)
SCOPE = 'edm-training3'
DATA_DIR = 'data/'

def get_base_price(asset_class):
    ranges = {
        'Common Stock': (random.uniform(10, 500), 0.02),
        'Corporate Bond': (random.uniform(95, 105), 0.005),
        'Government Bond': (random.uniform(95, 110), 0.003),
        'ETF': (random.uniform(20, 300), 0.015),
    }
    return ranges.get(asset_class, (100, 0.02))

instruments_to_quote = []
for filename, asset_class in [
    ('common-stock.csv', 'Common Stock'), ('investment-grade.csv', 'Corporate Bond'),
    ('government.csv', 'Government Bond'), ('etfs.csv', 'ETF'),
]:
    df = pd.read_csv(f'{DATA_DIR}{filename}')
    for _, row in df.iterrows():
        instruments_to_quote.append({
            'id': str(row['ClientInternal']).strip(),
            'asset_class': asset_class,
            'currency': str(row.get('Currency', 'USD')).strip()
        })

print(f"Quoting {len(instruments_to_quote)} instruments across 30 business days...")
dates = pd.bdate_range(start='2026-01-01', periods=30)
total_quotes = 0
for date in dates:
    date_str = date.strftime('%Y-%m-%dT00:00:00+00:00')
    batch = {}
    for instr in instruments_to_quote:
        base, vol = get_base_price(instr['asset_class'])
        price = round(base * (1 + random.uniform(-vol, vol)), 4)
        batch[f"{instr['id']}-{date.date()}"] = lm.UpsertQuoteRequest(
            quote_id=lm.QuoteId(
                quote_series_id=lm.QuoteSeriesId(
                    provider='Lusid', instrument_id=instr['id'],
                    instrument_id_type='ClientInternal',
                    quote_type='Price', field='mid'),
                effective_at=date_str),
            metric_value=lm.MetricValue(value=price, unit=instr['currency']))
    try:
        response = quotes_api.upsert_quotes(scope=SCOPE, request_body=batch)
        total_quotes += len(response.values)
    except lu.ApiException as e:
        print(f"  Error on {date.date()}: {e.reason}")
    if (dates.get_loc(date) + 1) % 10 == 0:
        print(f"  {dates.get_loc(date) + 1}/{len(dates)} dates processed...")

print(f"\n✅ NB05 Complete: {total_quotes} quotes upserted")